## Equal Width Binning


In [1]:
import pandas as pd
import numpy as np

# ─── Dataset: HDFC Loan Applicants ───
data = {
    'name':   ['Aarav','Priya','Ravi','Sneha','Karan',
                'Divya','Mohit','Ananya','Vikram','Pooja'],
    'age':    [23, 45, 31, 58, 27, 62, 38, 19, 52, 41],
    'income': [280000, 850000, 420000, 1200000, 310000,
               950000, 680000, 180000, 780000, 520000],
}
df = pd.DataFrame(data)

df

,name,age,income
0,Aarav,23,280000
1,Priya,45,850000
2,Ravi,31,420000
3,Sneha,58,1200000
4,Karan,27,310000
5,Divya,62,950000
6,Mohit,38,680000
7,Ananya,19,180000
8,Vikram,52,780000
9,Pooja,41,520000


In [2]:
df.age.describe()

,age
count,10.000000
mean,39.600000
std,14.758802
min,19.000000
25%,28.000000
50%,39.500000
75%,50.250000
max,62.000000


In [3]:
# ════════════════════════════════════════
# METHOD 1: pd.cut() — Custom Bin Edges (domain knowledge)
# ════════════════════════════════════════
# Age bins with custom boundaries
df['age_group'] = pd.cut(
    df['age'],
    bins=[0, 30, 45, 60, 100],              # Boundary points
    labels=['Young', 'Middle', 'Senior', 'Elder'],  # Bin names
    right=True                               # Includes right endpoint
)

print("\nBin Distribution:")
print(df['age_group'].value_counts())
print("Age Binning (Custom Edges):")
df[['name', 'age', 'age_group']]


Bin Distribution:
age_group
Middle    4
Young     3
Senior    2
Elder     1
Name: count, dtype: int64
Age Binning (Custom Edges):


,name,age,age_group
0,Aarav,23,Young
1,Priya,45,Middle
2,Ravi,31,Middle
3,Sneha,58,Senior
4,Karan,27,Young
5,Divya,62,Elder
6,Mohit,38,Middle
7,Ananya,19,Young
8,Vikram,52,Senior
9,Pooja,41,Middle


In [4]:
# ════════════════════════════════════════
# METHOD 2: pd.cut() — Auto Equal Width (n bins)
# sklearn decides boundaries automatically
# ════════════════════════════════════════
df['income_bin'] = pd.cut(
    df['income'],
    bins=3,                                  # 3 equal-width bins
    labels=['Low', 'Medium', 'High']
)

print("\nIncome Binning (3 Equal-Width Bins):")
df[['name', 'income', 'income_bin']]


Income Binning (3 Equal-Width Bins):


,name,income,income_bin
0,Aarav,280000,Low
1,Priya,850000,Medium
2,Ravi,420000,Low
3,Sneha,1200000,High
4,Karan,310000,Low
5,Divya,950000,High
6,Mohit,680000,Medium
7,Ananya,180000,Low
8,Vikram,780000,Medium
9,Pooja,520000,Low


In [5]:
# ════════════════════════════════════════
# METHOD 3: sklearn KBinsDiscretizer (Equal Width)
# ML Pipeline ke saath use karo!
# ════════════════════════════════════════
from sklearn.preprocessing import KBinsDiscretizer

kbd = KBinsDiscretizer(
    n_bins=4,
    encode='ordinal',     # 0,1,2,3 integers (ordinal) ya onehot
    strategy='uniform'    # equal width bins
)

df[['age_binned_sk']] = kbd.fit_transform(df[['age']])

print("\nBin edges learned:", kbd.bin_edges_)
print("\nsklearn KBinsDiscretizer (Equal Width):")
df[['name', 'age', 'age_binned_sk']]


Bin edges learned: [array([19.  , 29.75, 40.5 , 51.25, 62.  ])]

sklearn KBinsDiscretizer (Equal Width):


,name,age,age_binned_sk
0,Aarav,23,0.0
1,Priya,45,2.0
2,Ravi,31,1.0
3,Sneha,58,3.0
4,Karan,27,0.0
5,Divya,62,3.0
6,Mohit,38,1.0
7,Ananya,19,0.0
8,Vikram,52,3.0
9,Pooja,41,2.0


## Quantile Binning


In [6]:
from sklearn.preprocessing import KBinsDiscretizer

# ─── Dataset: Swiggy Restaurant Revenue (₹ Lakhs/month) ───

# Manual Data - preserve the size: 70 low, 20 mid, 10 high
revenue_low = np.array([0.5, 0.8, 1.0, 1.2, 1.5, 0.7, 0.9, 1.1, 1.3, 1.6, 0.6, 0.9, 1.1, 1.4, 1.7, 0.5, 0.8, 1.0, 1.2, 1.5,
                        0.7, 0.9, 1.1, 1.3, 1.6, 0.6, 0.9, 1.1, 1.4, 1.7, 0.5, 0.8, 1.0, 1.2, 1.5, 0.7, 0.9, 1.1, 1.3, 1.6,
                        0.6, 0.9, 1.1, 1.4, 1.7, 0.5, 0.8, 1.0, 1.2, 1.5, 0.7, 0.9, 1.1, 1.3, 1.6, 0.6, 0.9, 1.1, 1.4, 1.7,
                        0.5, 0.8, 1.0, 1.2, 1.5, 0.7, 0.9, 1.1, 1.3, 1.6]) # 70 values
revenue_mid = np.array([6.0, 8.5, 10.0, 12.5, 15.0, 7.5, 9.0, 11.0, 13.5, 16.0,
                        6.5, 8.0, 9.5, 11.5, 14.0, 7.0, 9.0, 10.5, 12.0, 14.5]) # 20 values
revenue_high = np.array([22.0, 28.0, 35.0, 42.0, 48.0, 25.0, 30.0, 38.0, 45.0, 49.0]) # 10 values

revenue = np.concatenate([revenue_low, revenue_mid, revenue_high])

df = pd.DataFrame({'restaurant_id': range(1, 101), 'revenue': revenue.round(2)})

print("Revenue Stats:")
print(df['revenue'].describe().round(2))

df

Revenue Stats:
count    100.00
mean       6.49
std       11.12
min        0.50
25%        0.90
50%        1.30
75%        8.12
max       49.00
Name: revenue, dtype: float64


,restaurant_id,revenue
0,1,0.5
1,2,0.8
2,3,1.0
3,4,1.2
4,5,1.5
...,...,...
95,96,25.0
96,97,30.0
97,98,38.0
98,99,45.0


In [7]:
# ════════════════════════════════════════
# METHOD 1: pd.qcut() — Quantile Binning
# ════════════════════════════════════════
df['revenue_tier_q'] = pd.qcut(
    df['revenue'],
    q=4,                                         # 4 quartiles (Q1, Q2, Q3, Q4)
    labels=['Budget', 'Standard', 'Premium', 'Elite']
)

print("\nQuantile Binning (pd.qcut):")
print(df['revenue_tier_q'].value_counts())
# Each tier should have ~25 restaurants!

df


Quantile Binning (pd.qcut):
revenue_tier_q
Budget      28
Elite       25
Standard    24
Premium     23
Name: count, dtype: int64


,restaurant_id,revenue,revenue_tier_q
0,1,0.5,Budget
1,2,0.8,Budget
2,3,1.0,Standard
3,4,1.2,Standard
4,5,1.5,Premium
...,...,...,...
95,96,25.0,Elite
96,97,30.0,Elite
97,98,38.0,Elite
98,99,45.0,Elite


In [8]:
# ════════════════════════════════════════
# METHOD 2: Compare Equal Width vs Quantile
# ════════════════════════════════════════
df['revenue_tier_ew'] = pd.cut(
    df['revenue'],
    bins=4,
    labels=['Budget', 'Standard', 'Premium', 'Elite']
)

print("\n=== Comparison: Equal Width vs Quantile ===")
print("Equal Width distribution:")
print(df['revenue_tier_ew'].value_counts())

print("\nQuantile distribution:")
print(df['revenue_tier_q'].value_counts())
# Quantile = balanced! Equal Width = mostly 'Budget'

df


=== Comparison: Equal Width vs Quantile ===
Equal Width distribution:
revenue_tier_ew
Budget      85
Standard     6
Elite        5
Premium      4
Name: count, dtype: int64

Quantile distribution:
revenue_tier_q
Budget      28
Elite       25
Standard    24
Premium     23
Name: count, dtype: int64


,restaurant_id,revenue,revenue_tier_q,revenue_tier_ew
0,1,0.5,Budget,Budget
1,2,0.8,Budget,Budget
2,3,1.0,Standard,Budget
3,4,1.2,Standard,Budget
4,5,1.5,Premium,Budget
...,...,...,...,...
95,96,25.0,Elite,Premium
96,97,30.0,Elite,Premium
97,98,38.0,Elite,Elite
98,99,45.0,Elite,Elite


In [9]:
# ════════════════════════════════════════
# METHOD 3: sklearn KBinsDiscretizer (strategy='quantile')
# ════════════════════════════════════════
kbd_q = KBinsDiscretizer(n_bins=4, encode='ordinal', strategy='quantile')
df[['revenue_binned']] = kbd_q.fit_transform(df[['revenue']])

print("\nsklearn KBinsDiscretizer (quantile) bin edges:")
print(kbd_q.bin_edges_[0].round(2))
# Bin widths alag hongi — lekin har bin mein ~25 rows!

df


sklearn KBinsDiscretizer (quantile) bin edges:
[ 0.5   0.9   1.3   8.12 49.  ]


,restaurant_id,revenue,revenue_tier_q,revenue_tier_ew,revenue_binned
0,1,0.5,Budget,Budget,0.0
1,2,0.8,Budget,Budget,0.0
2,3,1.0,Standard,Budget,1.0
3,4,1.2,Standard,Budget,1.0
4,5,1.5,Premium,Budget,2.0
...,...,...,...,...,...
95,96,25.0,Elite,Premium,3.0
96,97,30.0,Elite,Premium,3.0
97,98,38.0,Elite,Elite,3.0
98,99,45.0,Elite,Elite,3.0


## K-Means Binning


In [10]:
from sklearn.preprocessing import KBinsDiscretizer

# ─── Dataset: CIBIL Credit Score Data ───
# Natural clusters: Poor, Average, Good, Excellent credit

# Manual Data - preserve the size: 30 Poor, 40 Average, 20 Good, 10 Excellent
poor_scores = np.array([550, 560, 570, 580, 590, 565, 575, 585, 595, 555, 570, 580, 590, 560, 570, 580, 590, 550, 560, 570, 580, 590, 565, 575, 585, 595, 555, 570, 580, 590]) # 30 values
average_scores = np.array([650, 660, 670, 680, 690, 700, 655, 665, 675, 685, 695, 660, 670, 680, 690, 700, 650, 660, 670, 680, 690, 700, 655, 665, 675, 685, 695, 660, 670, 680, 690, 700, 650, 660, 670, 680, 690, 700, 655, 665]) # 40 values
good_scores = np.array([740, 750, 760, 770, 780, 745, 755, 765, 775, 785, 740, 750, 760, 770, 780, 745, 755, 765, 775, 785]) # 20 values
excellent_scores = np.array([800, 810, 820, 830, 840, 805, 815, 825, 835, 845]) # 10 values

scores = np.concatenate([
    poor_scores,
    average_scores,
    good_scores,
    excellent_scores
])

df = pd.DataFrame({'credit_score': scores})

print("Credit Score Stats:")
print(df['credit_score'].describe().round(1))

df

Credit Score Stats:
count    100.0
mean     677.2
std       83.8
min      550.0
25%      590.0
50%      675.0
75%      750.0
max      845.0
Name: credit_score, dtype: float64


,credit_score
0,550
1,560
2,570
3,580
4,590
...,...
95,805
96,815
97,825
98,835


In [11]:
# ════════════════════════════════════════
# METHOD: sklearn KBinsDiscretizer (strategy='kmeans')
# Easiest way — ek line mein K-Means binning!
# ════════════════════════════════════════
kbd_km = KBinsDiscretizer(
    n_bins=4,
    encode='ordinal',
    strategy='kmeans'      # K-Means algorithm use karo
)

df[['kmeans_bin']] = kbd_km.fit_transform(df[['credit_score']])

print("\nK-Means Binning (KBinsDiscretizer):")
print(df['kmeans_bin'].value_counts().sort_index())
print("\nK-Means bin edges:", kbd_km.bin_edges_[0].round(1))

df


K-Means Binning (KBinsDiscretizer):
kmeans_bin
0.0    30
1.0    40
2.0    20
3.0    10
Name: count, dtype: int64

K-Means bin edges: [550.  624.9 719.  792.5 845. ]


,credit_score,kmeans_bin
0,550,0.0
1,560,0.0
2,570,0.0
3,580,0.0
4,590,0.0
...,...,...
95,805,3.0
96,815,3.0
97,825,3.0
98,835,3.0


In [12]:
# ════════════════════════════════════════
# COMPARISON: All 3 methods side-by-side
# ════════════════════════════════════════
df['equal_width'] = pd.cut(df['credit_score'], bins=4,
                            labels=['Poor','Average','Good','Excellent'])
df['quantile']    = pd.qcut(df['credit_score'], q=4,
                             labels=['Poor','Average','Good','Excellent'])

print("\n=== Bin Distribution Comparison ===")
print("Equal Width:")
print(df['equal_width'].value_counts().sort_index())
print("\nQuantile:")
print(df['quantile'].value_counts().sort_index())
print("\nK-Means:")
print(df['kmeans_bin'].value_counts().sort_index())


=== Bin Distribution Comparison ===
Equal Width:
equal_width
Poor         30
Average      35
Good         19
Excellent    16
Name: count, dtype: int64

Quantile:
quantile
Poor         28
Average      23
Good         25
Excellent    24
Name: count, dtype: int64

K-Means:
kmeans_bin
0.0    30
1.0    40
2.0    20
3.0    10
Name: count, dtype: int64


## Binarization


In [13]:
from sklearn.preprocessing import Binarizer

# ─── Dataset: Apollo Diagnostics — Health Records ───
data = {
    'patient':      ['Aarav','Priya','Ravi','Sneha','Karan',
                     'Divya','Mohit','Ananya','Vikram','Pooja'],
    'glucose':      [95, 132, 87, 145, 118, 161, 78, 203, 109, 98],    # mg/dL
    'bmi':          [22.5, 28.3, 19.8, 31.4, 26.1, 34.2, 17.9, 38.5, 24.7, 21.3],
    'purchases':    [0, 5, 2, 0, 8, 1, 0, 3, 0, 6],   # Monthly purchases
}
df = pd.DataFrame(data)

print("Original Data:")
df

Original Data:


,patient,glucose,bmi,purchases
0,Aarav,95,22.5,0
1,Priya,132,28.3,5
2,Ravi,87,19.8,2
3,Sneha,145,31.4,0
4,Karan,118,26.1,8
5,Divya,161,34.2,1
6,Mohit,78,17.9,0
7,Ananya,203,38.5,3
8,Vikram,109,24.7,0
9,Pooja,98,21.3,6


In [14]:
# ════════════════════════════════════════
# METHOD 1: sklearn Binarizer
# Threshold se upar = 1, neeche = 0
# ════════════════════════════════════════

# Glucose: 126 mg/dL = diabetes threshold (WHO standard)
glucose_bin = Binarizer(threshold=126)
df['diabetic'] = glucose_bin.fit_transform(df[['glucose']])

# BMI: 25 = overweight threshold
bmi_bin = Binarizer(threshold=25)
df['overweight'] = bmi_bin.fit_transform(df[['bmi']])

print("\n--- sklearn Binarizer ---")
df[['patient', 'glucose', 'diabetic', 'bmi', 'overweight']]


--- sklearn Binarizer ---


,patient,glucose,diabetic,bmi,overweight
0,Aarav,95,0,22.5,0.0
1,Priya,132,1,28.3,1.0
2,Ravi,87,0,19.8,0.0
3,Sneha,145,1,31.4,1.0
4,Karan,118,0,26.1,1.0
5,Divya,161,1,34.2,1.0
6,Mohit,78,0,17.9,0.0
7,Ananya,203,1,38.5,1.0
8,Vikram,109,0,24.7,0.0
9,Pooja,98,0,21.3,0.0


In [15]:
# ════════════════════════════════════════
# METHOD 2: pandas — simple comparison (same result!)
# ════════════════════════════════════════
df['diabetic_pd']   = (df['glucose'] > 126).astype(int)
df['overweight_pd'] = (df['bmi'] > 25).astype(int)

# Binary feature: Kya customer ne koi purchase kiya?
df['is_buyer'] = (df['purchases'] > 0).astype(int)

print("\n--- pandas Binarization ---")
df[['patient', 'purchases', 'is_buyer']]


--- pandas Binarization ---


,patient,purchases,is_buyer
0,Aarav,0,0
1,Priya,5,1
2,Ravi,2,1
3,Sneha,0,0
4,Karan,8,1
5,Divya,1,1
6,Mohit,0,0
7,Ananya,3,1
8,Vikram,0,0
9,Pooja,6,1


In [16]:
# ════════════════════════════════════════
# Check: Diabetes prevalence in our sample
# ════════════════════════════════════════
print(f"\nDiabetic patients: {df['diabetic'].sum()}/{len(df)}")
print(f"Overweight patients: {df['overweight'].sum()}/{len(df)}")
print(f"Buyers: {df['is_buyer'].sum()}/{len(df)}")


Diabetic patients: 4/10
Overweight patients: 5.0/10
Buyers: 6/10


## For reference:

1. https://youtu.be/ui9UwDYFvcM